&nbsp;
&nbsp;
![](../_resources/images/e2eai-5.jpg)


# Generative AI with Databricks

## From Predictive to Prescriptive Maintenance
Manufacturers face labor shortages, supply chain disruptions, and rising costs, making efficient maintenance essential. Despite investments in maintenance programs, many struggle to boost asset productivity due to technician shortages and poor knowledge-sharing systems. This leads to knowledge loss and operational inefficiencies.

<div style="font-family: 'DM Sans';">
  <div style="width: 400px; color: #1b3139; margin-left: 50px; margin-right: 50px; float: left;">
    <div style="color: #ff5f46; font-size:50px;">73%</div>
    <div style="font-size:25px; margin-top: -20px; line-height: 30px;">
      of manufacturers struggle to recruit maintenance technicians — McKinsey (2023)
    </div>
    <div style="color: #ff5f46; font-size:50px;">55%</div>
    <div style="font-size:25px; margin-top: -20px; line-height: 30px;">
      of manufacturers lack formal knowledge-sharing systems — McKinsey (2023)
    </div>
  </div>
</div>

Generative AI can transform maintenance by reducing downtime and improving productivity. While predictive maintenance anticipates failures, Generative AI enables prescriptive maintenance. Using historical data, AI systems can identify issues, generate solutions, and assist technicians, allowing junior staff to perform effectively and freeing experts for complex tasks.
<br><br>

### From Models to Agent Systems
Generative AI is moving from standalone models to modular agent systems ([Zaharia et al., 2024](https://bair.berkeley.edu/blog/2024/02/18/compound-ai-systems/)). These systems integrate retrievers, models, prompts, and tools to handle complex tasks. Their modular design allows seamless upgrades (e.g., integrating a new LLM) and adaptation to changing needs.

<br>
<img style="float: right; margin-top: 10px;" width="700px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/team_flow_liza.png" />

<br>
<!div style="font-size: 19px; margin-left: 0px; clear: left; padding-top: 10px; ">

**Databricks empowers users with a Data + AI platform for Prescriptive Maintenance.** 
Let’s explore how to deploy this in production.
<br><br>
<div style="font-size: 19px; margin-left: 0px; clear: left; padding-top: 10px; ">
<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/liza.png" style="width:80px">
<br>
<h3 style="padding: 10px 0px 0px 5px;">Liza, a Generative AI engineer, uses the Databricks Intelligence Platform to:</h3>
<ul style="list-style: none; padding: 0; margin-left: 05%;">
  <li style="margin-bottom: 10px; display: flex; align-items: center;">
    <div class="badge" style="height: 30px; width: 30px; border-radius: 50%; background: #fcba33; color: white; text-align: center; line-height: 30px; font-weight: bold; margin-right: 10px;">1</div>
    Build real-time data pipelines
  </li>
  <li style="margin-bottom: 10px; display: flex; align-items: center;">
    <div class="badge" style="height: 30px; width: 30px; border-radius: 50%; background: #fcba33; color: white; text-align: center; line-height: 30px; font-weight: bold; margin-right: 10px;">2</div>
    Retrieve vectors & features
  </li>
  <li style="margin-bottom: 10px; display: flex; align-items: center;">
    <div class="badge" style="height: 30px; width: 30px; border-radius: 50%; background: #fcba33; color: white; text-align: center; line-height: 30px; font-weight: bold; margin-right: 10px;">3</div>
    Create AI agent tools
  </li>
  <li style="margin-bottom: 10px; display: flex; align-items: center;">
    <div class="badge" style="height: 30px; width: 30px; border-radius: 50%; background: #fcba33; color: white; text-align: center; line-height: 30px; font-weight: bold; margin-right: 10px;">4</div>
    Build & deploy agents
  </li>
  <li style="margin-bottom: 10px; display: flex; align-items: center;">
    <div class="badge" style="height: 30px; width: 30px; border-radius: 50%; background: #fcba33; color: white; text-align: center; line-height: 30px; font-weight: bold; margin-right: 10px;">5</div>
    Operate in batch or real-time
  </li>
  <li style="display: flex; align-items: center;">
    <div class="badge" style="height: 30px; width: 30px; border-radius: 50%; background: #fcba33; color: white; text-align: center; line-height: 30px; font-weight: bold; margin-right: 10px;">6</div>
    Evaluate agent performance
  </li>
</ul>
</div>

<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=4003492105941350&notebook=%2F05-Generative-AI%2F05.1-ai-tools-iot-turbine-prescriptive-maintenance&demo_name=lakehouse-iot-platform&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-iot-platform%2F05-Generative-AI%2F05.1-ai-tools-iot-turbine-prescriptive-maintenance&version=1">

## Building Agent Systems with Databricks Mosaic AI agent framework

We will build an Agent System designed to generate prescriptive work orders for wind turbine maintenance technicians. This system integrates multiple interacting components to ensure proactive and efficient maintenance, thereby optimizing the overall equipment effectiveness.

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/iot_agent_graph_v2_0.png" style="margin-left: 5px; float: right"  width="1000px;">

Databricks simplifies this by providing a built-in service to:

- Create and store your AI tools leveraging UC functions
- Execute the AI tools in a safe way
- Use agents to reason about the tools you selected and chain them together to properly answer your question. 


This notebook creates the three Mosaic AI tools and associated Mosaic AI endpoints, which will be composed together into a agent in notebook [05.2-agent-creation-guide]($./05.2-agent-creation-guide).
1. **Turbine specifications retriever** which retrieve the turbine specifications based on its id.
2. **Turbine maintenance predictor** which uses a Model Serving endpoint to predict turbines at risk of failure.
3. **Turbine maintenance guide**  which uses a Vector Search endpoint to retrieve maintenance guide based on the turbines and issues being adressed.

In [0]:
%pip install databricks-vectorsearch==0.49 databricks-feature-engineering==0.8.0 databricks-sdk==0.40.0 sentence-transformers==3.0.1
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../_resources/00-setup $reset_all_data=false

## Configuration file

Please change your catalog and schema here to run the demo on a different catalog.

 
<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=4003492105941350&notebook=%2Fconfig&demo_name=lakehouse-iot-platform&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-iot-platform%2Fconfig&version=1">


# Technical Setup notebook. Hide this cell results
Initialize dataset to the current user and cleanup data when reset_all_data is set to true

Do not edit

In [0]:
from pyspark.sql.functions import col, pandas_udf, concat, lit
from pyspark.sql.types import ArrayType, DoubleType
import pandas as pd
from sentence_transformers import SentenceTransformer

USE CATALOG `main`
using catalog.database `main`.`e2eai_iot_turbine`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


## Part 1: Create the Turbine Specification Retriever as a tool to return sensor readings for a turbine

Edit the FROM table if you changed from the default catalog/schema in your config file.

In [0]:
%sql
DROP FUNCTION IF EXISTS turbine_specifications_retriever;

--turbine_specifications_retriever to get the current status of a turbine
--This function is used to retrieve the turbine specifications based on its id

CREATE OR REPLACE FUNCTION 
turbine_specifications_retriever(turbine_id STRING COMMENT 'ID of the wind turbine to look up')
RETURNS TABLE (
  avg_energy DOUBLE COMMENT 'Average energy reading',
  std_sensor_A DOUBLE COMMENT 'Sensor A reading',
  std_sensor_B DOUBLE COMMENT 'Sensor B reading',
  std_sensor_C DOUBLE COMMENT 'Sensor C reading',
  std_sensor_D DOUBLE COMMENT 'Sensor D reading',
  std_sensor_E DOUBLE COMMENT 'Sensor E reading',
  std_sensor_F DOUBLE COMMENT 'Sensor F reading'
)
LANGUAGE SQL
COMMENT 'This function retrieves the turbine sensor readings / specifications based on the turbine_id'
RETURN
(
SELECT 

avg_energy, std_sensor_A, std_sensor_B, std_sensor_C, std_sensor_D, std_sensor_E, std_sensor_F
FROM main.e2eai_iot_turbine.turbine_current_features
WHERE turbine_id = turbine_specifications_retriever.turbine_id
SORT BY hourly_timestamp DESC
limit 1
);

Now, test our tool:

In [0]:
%sql
SELECT * FROM turbine_specifications_retriever('004a641f-e9e5-9fff-d421-1bf88319420b')

avg_energy,std_sensor_A,std_sensor_B,std_sensor_C,std_sensor_D,std_sensor_E,std_sensor_F
0.074818609038791,1.058048335093487,2.4852932716249665,2.8927160852893152,2.1567050955955853,2.2120358529793696,5.614526027139428


## Part 2: Create the Turbine Predictor as a tool to predict turbine failure

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/iot_agent_graph_v2_1.png" style="float: right; width: 600px; margin-left: 10px">

To enable our Agent System to predict turbine failtures based on industrial IoT sensor readings, we will rely on the model we deployed previously in the  [./04.3-running-inference-iot-turbine]($./04.3-running-inference-iot-turbine) notebook. 

**Make sure you run this ML notebook to create the model serving endpoint!**


### Using the Model Serving as tool to predict faulty turbines
Let's define the turbine predictor tool function our LLM agent will be able to execute. 

AI agents use [AI Agent Tools](https://docs.databricks.com/en/generative-ai/create-log-agent.html#create-ai-agent-tools) to perform actions besides language generation, for example to retrieve structured or unstructured data, execute code, or talk to remote services (e.g. send an email or Slack message). 

These functions can contain any logic, from simple SQL to advanced python. Below we wrap the model serving endpoint in a SQL function using '[ai_query](https://docs.databricks.com/en/sql/language-manual/functions/ai_query.html)' function, as we tested in the previous notebook.

In [0]:
%sql
DROP FUNCTION IF EXISTS turbine_maintenance_predictor;

--Use turbine_maintenance_predictor to get a prediction of whether or not a turbine sensor is faulty to facilitate proactive maintenance
--This function is used to predict turbine maintenance based on energy and sensor readings

CREATE OR REPLACE FUNCTION 
turbine_maintenance_predictor(avg_energy DOUBLE, 
                              std_sensor_A DOUBLE, 
                              std_sensor_B DOUBLE, 
                              std_sensor_C DOUBLE, 
                              std_sensor_D DOUBLE, 
                              std_sensor_E DOUBLE, 
                              std_sensor_F DOUBLE
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'This tool predicts whether or not a turbine is faulty to facilitate proactive maintenance. It expects an array of 7 double values (energy and sensor readings) as input and returns a string indicating which sensor is predicted to be faulty or if all sensors are ok.'
RETURN
(
    SELECT 
        -- The xgboost model returns a float; translate it back to a string
        CASE WHEN float_prediction=0 THEN "F"
            WHEN float_prediction=1 THEN "ok"
            WHEN float_prediction=2 THEN "B"
            WHEN float_prediction=3 THEN "D"
        ELSE "faulty" END AS prediction    
    FROM (
    SELECT ai_query('e2eai_iot_turbine_prediction_endpoint',
        STRUCT(avg_energy AS avg_energy,
            std_sensor_A AS std_sensor_A,
            std_sensor_B AS std_sensor_B,
            std_sensor_C AS std_sensor_C,
            std_sensor_D AS std_sensor_D,
            std_sensor_E AS std_sensor_E,
            std_sensor_F AS std_sensor_F
        ),
        'FLOAT'
    ) AS float_prediction)
);

Now, test our tool.

In [0]:
%sql
SELECT turbine_maintenance_predictor(
  0.9000803742589635,                           -- avg_energy
  2.2081154200781867,                           -- std_sensor_A
  2.6012126574143823,                           -- std_sensor_B
  2.1075958066966423,                           -- std_sensor_C
  2.2081154200781867,                           -- std_sensor_D
  2.6012126574143823,                           -- std_sensor_E
  2.1075958066966423                            -- std_sensor_F
) AS prediction

prediction
ok


Now build an alternate tool using the python code we just tested. 

**You will need to add an API TOKEN and API ROOT before running this code.** The notebook API_TOKEN that we used for testing the python code above will not work.  Instead, create a [Personal Access Token](https://docs.databricks.com/aws/en/dev-tools/auth/pat).

In [0]:
# %sql
# CREATE OR REPLACE FUNCTION turbine_maintenance_predictor(sensor_values ARRAY<DOUBLE>)
# RETURNS STRING
# LANGUAGE PYTHON
# COMMENT 'This tool predicts whether or not a turbine is faulty to facilitate proactive maintenance. It expects an array of 7 double values (energy and sensor readings) as input and returns a string indicating if a particular sensor is predicted to be faulty or if all sensors are ok.'
# AS 
# $$

# import numpy as np
# import pandas as pd
# import json 
# import requests

# #API TOKEN AND URL HERE

# api_token = ""
# api_root = ""

# model_serving_endpoint_name = 'e2eai_iot_turbine_prediction_endpoint'

# columns = ['avg_energy', 'std_sensor_A', 'std_sensor_B', 'std_sensor_C', 'std_sensor_D', 'std_sensor_E', 'std_sensor_F']

# samp_ar = np.array([sensor_values])

# data = pd.DataFrame(samp_ar, columns=columns)

# url = f'{api_root}/serving-endpoints/{model_serving_endpoint_name}/invocations'

# headers = {'Authorization': f'Bearer {api_token}', 
#             'Content-Type': 'application/json'}


# ds_dict = {'dataframe_split': data.to_dict(orient='split')} if isinstance(data, pd.DataFrame) else tf_serving_json

# data_json = json.dumps(ds_dict, allow_nan=True)

# response = requests.request(method='POST', headers=headers, url=url, data=data_json)

# if response.status_code != 200:
#     raise Exception(f'Request failed with status {response.status_code}, {response.text}')

# if response.json()['predictions'][0] == 0:
#     return 'Sensor F fault'
# elif response.json()['predictions'][0] == 1:
#     return 'ok'
# elif response.json()['predictions'][0] == 2:
#     return 'Sensor B fault'
# elif response.json()['predictions'][0] == 3:
#     return 'Sensor D fault'
# else:
#     return 'faulty'

# $$ 


Test the python/SQL tool.

In [0]:
# %sql
# SELECT turbine_maintenance_predictor(array(0.1889792, 
#                                            0.9644652, 
#                                            2.65583866, 
#                                            3.4528106, 
#                                            2.48515875,
#                                            2.28840325, 
#                                            4.70213899)) as prediction

Our agent should call turbine_specifications_retriever() to get sensor readings, then call turbine_maintenance_predictor() to get a prediction.

## Part 3: Add a tool to access our maintenance guide content and provide support to the operator during maintenance operation

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/iot_agent_graph_v2_3.png" style="float: right; width: 600px; margin-left: 10px">


We were provided with PDF guide containing all the error code and maintenance steps for the critical components of our wind turbine. The're saved as pdf file in our volume.

Let's parse them and index them so that we can properly retrieve them. We'll save them in a Vector Search endpoint and leverage it to guide the operators with the maintenance step and recommendations.

We'll use a Managed embedding index to make it simple. In this section we will:

1. Parse and save our PDF text in a Delta Table using Databricks AI Query `ai_parse_document`
2. Create a `Vector Search endpoint` (required to host your vector search index)
3. Create a `Vector Search Direct Index`  (the actual index)
4. Create a `Tool (UC function)` using our vector search 


### 2.1. Parse and save our PDF text
Let's start by parsing the maintenance guide documents, saved as pdf in our volume:

In [0]:
%sql
DROP TABLE IF EXISTS turbine_maintenance_guide;

CREATE TABLE turbine_maintenance_guide (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  EAN STRING,
  weight STRING,
  component_type STRING,
  component_name STRING,
  full_guide STRING,
  embedding ARRAY<DOUBLE> COMMENT 'Precomputed embedding vector for full_guide')
  TBLPROPERTIES (delta.enableChangeDataFeed = true);

Extract specific features from each maintenance manual.

Edit the FROM table if you changed from the default catalog/schema in your config file.

This query will take 10+ minutes to run.

### Alternative Approach: Local Embedding Computation

Since no embedding endpoints are available in this workspace, we'll compute embeddings **locally on the cluster** using the `sentence-transformers` library.

**How it works:**
1. Install `sentence-transformers` package
2. Create a Python UDF that loads the `all-MiniLM-L6-v2` model and generates embeddings
3. Use this UDF in our INSERT query instead of `ai_query()`

**Benefits:**
* ✓ No endpoint required
* ✓ Runs on your existing cluster compute
* ✓ Uses a high-quality open-source embedding model (384 dimensions)

**Note:** This will be slower than using a managed endpoint, but it works without any additional infrastructure.

In [0]:

# Load the embedding model (will be broadcast to workers)
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Define a Pandas UDF for embedding computation
@pandas_udf(ArrayType(DoubleType()))
def compute_embedding_udf(texts: pd.Series) -> pd.Series:
    # Generate embeddings for a batch of texts
    embeddings = model.encode(texts.tolist())
    return pd.Series([emb.tolist() for emb in embeddings])

# Read and parse PDFs, then compute embeddings
df_with_embeddings = (
    spark.sql("""
        SELECT 
            ai_query('databricks-gemma-3-12b',
                CONCAT("Extract the EAN from the following text. Return only the EAN. \\n\\nText:", full_guide)
            ) AS EAN,
            ai_query('databricks-gemma-3-12b',
                CONCAT("Extract the component weight from the following text. Return only the component weight. \\n\\nText:", full_guide)
            ) AS weight,
            ai_query('databricks-gemma-3-12b',
                CONCAT("Extract the component type from the following text. Return only the component type. \\n\\nText:", full_guide)
            ) AS component_type,
            ai_query('databricks-gemma-3-12b',
                CONCAT("Extract the component name from the following text. Return only the component name. \\n\\nText:", full_guide)
            ) AS component_name,
            full_guide
        FROM (
            SELECT array_join(
                transform(
                    parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>,
                    x -> x.content
                ), 
                '\\n'
            ) AS full_guide
            FROM (
                SELECT ai_parse_document(content) AS parsed_document
                FROM READ_FILES("/Volumes/main/e2eai_iot_turbine/e2eai_turbine_raw_landing/maintenance_guide", format => 'binaryFile')
            )
        )
    """)
    .withColumn("embedding", compute_embedding_udf(col("full_guide")))
)

# Insert into the table
df_with_embeddings.write.mode("append").saveAsTable("turbine_maintenance_guide")

print(f"Inserted {df_with_embeddings.count()} rows with embeddings into turbine_maintenance_guide")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Inserted 18 rows with embeddings into turbine_maintenance_guide


The [`ai_parse_document()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_parse_document) function invokes a state-of-the-art generative AI model from Databricks Foundation Model APIs to extract structured content from unstructured documents.

In [0]:
%sql
SELECT * FROM turbine_maintenance_guide
limit 10

id EAN weight component_type component_name full_guide embedding 1 96356234 4014g Anemometer AeroSense VTX-220 AeroSense VTX-220 Precision Anemometer
Professional Maintenance Manual - Wind Turbine Component
Part Type: Anemometer
EAN: 96356234
Compatible Turbine: Skylance XR550 Coastal Turbine Platform
Sensors Used: sensor_D, sensor_C
Dimensions: 364mm - 286mm
Weight: 4014g
Stock Zone: America/New_York
Component Overview
The AeroSense VTX-220 is a precision optical-cup anemometer designed to operate in harsh
marine and coastal environments.
It delivers high-frequency wind speed measurements to the turbine-s main control unit via sensor_D
and communicates backup diagnostics via sensor_C.
Equipped with a UV-resistant ABS housing and dual-bearing stainless steel shaft, it maintains
sub-0.1s response latency in gusts up to 180 km/h.
Recognizing Issues
Operators may notice discrepancies between actual weather data and turbine logs, delayed SCADA
wind alerts, or sudden drops in power generation.
Typical warning signs include inconsistent wind-speed readings or visible physical obstruction of the rotor cups.
Error Codes & Troubleshooting
ANM-100
Description: Low RPM detected despite wind presence. Likely mechanical blockage or bearing failure.
Resolution: Inspect the cup rotor for salt deposits or bird interference. Apply approved cleaning fluid
and rotate manually to check bearing friction.
ANM-201
Description: Intermittent signal loss on sensor_D channel. Data irregularities exceed 15% over 10 minutes.
Resolution: Replace connector or test with alternate input pin. Ensure waterproof sealing and
corrosion-free contact surface.
ANM-310
Description: Excessive vibration detected beyond 2.0g. Shaft imbalance or mounting flange fault suspected.
Resolution: Use vibration sensor diagnostics to assess severity. Tighten mounting base, and
recalibrate pitch compensation in firmware settings.
Recommended Maintenance Schedule
Inspect every 2,500 operational hours. Replace every 12,000 hours or if more than two error events occur in a 90-day period.
Preventative cleaning every 6 months in salt-air environments is strongly advised.
Certified Maintenance Procedure
1. Engage turbine maintenance mode from SCADA dashboard. Confirm rotor lock and system isolation.
2. Access the rooftop nacelle via secured ladder or lift. Wear high-visibility PPE and fall arrest gear rated for 100kg minimum.
3. Locate the anemometer mast on the rear left corner of the nacelle shell. Document physical condition with photos before removal.
4. Disconnect the twin sensor cable junction box under the mast. Use an IP67-rated cap to cover open leads during maintenance.
5. Using a torque wrench, loosen the 3 mounting bolts (13mm) securing the anemometer base. Hold the unit from above to prevent fall.
6. Gently lift the AeroSense VTX-220 unit upward and inspect the shaft coupling area for corrosion
or fatigue.
7. Clean the mounting flange area with alcohol-based surface cleaner and dry thoroughly using lint-free cloth.
8. Prepare the replacement unit by verifying serial number, firmware revision, and alignment pin compatibility.
9. Insert the VTX-220 anemometer onto the flange, align the guiding notch, and secure all bolts to
28 Nm torque.
10. Reconnect sensor_D and sensor_C to the color-coded terminal blocks in the junction box. Use dielectric grease on terminals.
11. Run a full sensor check using the Skylance Diagnostic Utility. Validate RPM, signal variance,
and latency under simulated gust input.
12. Recalibrate the wind-speed baseline against local weather station data. Accept results only if variance < 1.2%.
13. Document replacement with timestamp, technician ID, and attach photos to maintenance log in the turbine management system. List(-0.026319775730371475, 0.03922226279973984, 0.015073644928634167, 0.018562594428658485, 0.07579002529382706, -0.035113707184791565, 0.03522593900561333, 0.02798393927514553, 0.0458342470228672, 0.0012557081645354629, 0.04847488924860954, -

### 2.2. Creating the Vector Search endpoint

Let's create a new Vector search endpoint. You can also use the [UI under Compute](#/setting/clusters/vector-search) to directly create your endpoint.

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True) #Whether to disable authentication notice messages. Default is False.

if not endpoint_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME):
    vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME, 
                        endpoint_type="STANDARD")

wait_for_vs_endpoint_to_be_ready(vsc, 
                                 VECTOR_SEARCH_ENDPOINT_NAME)


print(f"Endpoint named {VECTOR_SEARCH_ENDPOINT_NAME} is ready.")

Endpoint named e2eai_vs_endpoint is ready.



### 2.3 Creating the Vector Search Index

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/index_creation.gif?raw=true" width="600px" style="float: right; margin-left: 10px">

You can view your endpoint on the [Vector Search Endpoints UI](#/setting/clusters/vector-search). Click on the endpoint name to see all indexes that are served by the endpoint.

All we now have to do is to as Databricks to create the index on top of our table. The Delta Table will automatically be synched with the index.


Again, you can do that using your Unity Catalog UI, and selecting the turbine_maintenance_guide table in your Unity Catalog, and click on add a vector search. 

In [0]:
import databricks.sdk.service.catalog as c

# Where we want to store our index
vs_index_fullname = f"{catalog}.{db}.turbine_maintenance_guide_vs_index"

if not index_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname):
  print(f"Creating index {vs_index_fullname} on endpoint {VECTOR_SEARCH_ENDPOINT_NAME}...")
  
  index = vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    source_table_name=f"{catalog}.{db}.turbine_maintenance_guide",
    index_name=vs_index_fullname,
    pipeline_type="TRIGGERED",
    primary_key='id',
    embedding_source_column="full_guide",
    # Use precomputed embeddings instead of managed embedding endpoint
    embedding_vector_column="embedding",
    embedding_dimension=384  # all-MiniLM-L6-v2 produces 384-dimensional vectors
  )
else:
  print(f"Grabbing existing index {vs_index_fullname} on endpoint {VECTOR_SEARCH_ENDPOINT_NAME}...")
  index = vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)

# Note: With precomputed embeddings using local Python UDF:
# 1. No endpoint required - embeddings computed on cluster
# 2. Store embeddings in a column (embedding_vector_column)
# 3. Specify the embedding dimension explicitly (384 for all-MiniLM-L6-v2)
# 4. Ensure all embeddings have the same dimension

Creating index main.e2eai_iot_turbine.turbine_maintenance_guide_vs_index on endpoint e2eai_vs_endpoint...


In [0]:
exists = index_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
display(spark.createDataFrame([(vs_index_fullname, exists)], ["index_name", "exists"]))

index_name,exists
main.e2eai_iot_turbine.turbine_maintenance_guide_vs_index,true


### 2.4 Create our tool
Below, we utilize the _VECTOR\_SEARCH_ SQL function from Databricks to easily set up our maintenance reports retriever function. Our agent will utilize this function in the subsequent steps!

The [vector_search()](https://docs.databricks.com/aws/en/sql/language-manual/functions/vector_search) function is a SQL AI function that queries a Mosaic AI Vector Search index directly from SQL. It performs similarity search (and optionally hybrid keyword+vector search) against an index and returns the top matching rows with selected columns.

In [0]:
spark.sql("DROP FUNCTION IF EXISTS turbine_maintenance_guide_retriever")

spark.sql(f"""
CREATE OR REPLACE FUNCTION 
turbine_maintenance_guide_retriever(question STRING COMMENT 'Question to be answered from the turbine maintenance guides.')
RETURNS ARRAY<STRING>
LANGUAGE PYTHON
COMMENT 'This tool searches/retrieves the wind turbine maintenance guide for a given question using precomputed embeddings'
AS $$
import sys
import subprocess

# Install required packages in the function execution environment
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers==3.0.1", "databricks-vectorsearch"])

import json
from sentence_transformers import SentenceTransformer
from databricks.vector_search.client import VectorSearchClient

# Initialize model and client
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
vsc = VectorSearchClient(disable_notice=True)

# Compute embedding for the question
query_embedding = embedding_model.encode(question).tolist()

# Search the vector index
index = vsc.get_index(
    endpoint_name='{VECTOR_SEARCH_ENDPOINT_NAME}',
    index_name='{catalog}.{db}.turbine_maintenance_guide_vs_index'
)

results = index.similarity_search(
    query_vector=query_embedding,
    columns=["full_guide"],
    num_results=1
)

# Extract and return guide text
if results and 'result' in results and 'data_array' in results['result']:
    guides = [row[0] for row in results['result']['data_array']]
    return guides
else:
    return []
$$
""")

print("✓ Created persistent UC function: turbine_maintenance_guide_retriever")
print(f"  Location: {catalog}.{db}.turbine_maintenance_guide_retriever")
print("\n✓ On serverless compute, the function installs its own dependencies automatically.")
print("  No cluster library configuration needed!")
print("\nNote: First execution will be slower due to package installation, but subsequent calls will be cached.")

✓ Created persistent UC function: turbine_maintenance_guide_retriever
  Location: main.e2eai_iot_turbine.turbine_maintenance_guide_retriever

⚠️  IMPORTANT: For this function to work, 'sentence-transformers' must be installed as a cluster library.
   Current installation via %pip only affects the notebook environment, not UC function executors.

   To install as cluster library:
   1. Go to Compute in the left sidebar
   2. Select your cluster
   3. Click 'Libraries' tab
   4. Click 'Install New'
   5. Select 'PyPI' and enter: sentence-transformers==3.0.1
   6. Click 'Install'
   7. Restart the cluster after installation


In [0]:
%sql

-- understanding collect_list() function
--SELECT collect_list(col) FROM VALUES (1), (2), (NULL), (1) AS tab(col);
 
-- returns [1,2,1]


In [0]:
%sql 
-- Let's test the tool we created
SELECT turbine_maintenance_guide_retriever('The VibeGuard TVS-950 is giving me an error code TVS-001.') AS reports

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-431795733929430>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "\n-- Let's test the tool we created\nSELECT turbine_maintenance_guide_retriever('The VibeGuard TVS-950 is giving me an error code TVS-001.') AS reports\n")

File /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:1

## Exploring Mosaic AI Tools in Unity Catalog

Our tools are ready! 

You can now view the UC function tools in Catalog Explorer. Click **Catalog** in the sidebar. In the Catalog Explorer, navigate to your catalog and schema. 

The UC function tools appears under **Functions**. 

<img src="https://github.com/Datastohne/demo/blob/main/Screenshot%202024-09-18%20at%2016.24.24.png?raw=true"/>

## What’s next: test your Agents with Databricks Playground

Now that we have our AI Tools ready and registered in Unity Catalog, we can compose them into an agent system that generates maintenance work orders using the Mosaic AI agent framework.

Open the [05.2-agent-creation-guide]($./05.2-agent-creation-guide) notebook to create and deploy the system.

In [0]:
df = spark.sql(f"SELECT * FROM main.e2eai_iot_turbine.turbine_hourly_features LIMIT 10")
display(df)

turbine_id,hourly_timestamp,avg_energy,std_sensor_A,std_sensor_B,std_sensor_C,std_sensor_D,std_sensor_E,std_sensor_F,location,model,state,abnormal_sensor
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T17:00:00.000Z,0.18897920400916973,0.9644652043128558,2.6558386572409103,3.4528106013576214,2.485158752607405,2.2884032468369284,4.702138990110717,Tupelo,EpicWind,America/Chicago,sensor_F
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T18:00:00.000Z,0.19212257629921775,1.0681855556261903,2.3848184303882847,3.303412042721332,2.172251292324001,2.342593019596896,4.870875418724548,Tupelo,EpicWind,America/Chicago,sensor_F
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T19:00:00.000Z,0.1735634457450677,1.1420887720146298,2.062708699095104,3.019329663712003,2.339552044868049,2.7306978700770164,4.237196637787606,Tupelo,EpicWind,America/Chicago,sensor_F
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T20:00:00.000Z,0.10343409262714734,1.0498727154061804,2.2192165091594975,3.246726138931612,2.3204665834317817,2.662700177613455,4.289404582190178,Tupelo,EpicWind,America/Chicago,sensor_F
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T21:00:00.000Z,0.15481243527493338,1.0325552090494656,2.142101655549623,2.7298423212662217,2.3597486817214515,2.761466398058171,4.588788770497015,Tupelo,EpicWind,America/Chicago,sensor_F
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T22:00:00.000Z,0.0847723255024208,1.0021697211227565,2.0968943765292085,2.9215472587753415,2.477840322666964,2.9466029618007314,4.357159925464822,Tupelo,EpicWind,America/Chicago,sensor_F
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T23:00:00.000Z,0.074818609038791,1.058048335093487,2.4852932716249665,2.8927160852893152,2.1567050955955853,2.2120358529793696,5.614526027139428,Tupelo,EpicWind,America/Chicago,sensor_F
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T17:00:00.000Z,0.12839653721057284,1.0656088831997519,1.9263319253102174,3.3330563526547747,2.2300401961414615,2.354626086386649,1.8913049031607982,Crystal Lake,EpicWind,America/Chicago,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T18:00:00.000Z,0.8542245491303897,1.080309777815946,1.9618452098136365,2.9717426105145472,2.306627597988137,2.5166973688595817,1.980452948870913,Crystal Lake,EpicWind,America/Chicago,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T19:00:00.000Z,0.4915535666395597,1.0646332592567707,2.2186746553400307,3.3459438407963433,2.2847856939507167,2.5560343320959498,1.9519204325253467,Crystal Lake,EpicWind,America/Chicago,ok
